In [1]:
import sys
sys.path.append('../')

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [4]:
import torch
import math
import pandas as pd

from src.config.paths import BILINGUAL_PATH
from src.core.speech import BiASR

In [5]:
df = pd.read_parquet(BILINGUAL_PATH / 'libri-train.parquet')

In [6]:
asr = BiASR()
asr.model.print_params()

CTC Network	 1.90M  	(7.24MB) 
Conformer	 33.56M  	(128.02MB) 
Prediction Net	 2.47M  	(9.43MB) 
Link Network	 0.18M  	(0.69MB) 
Joint Network	 3.08M  	(11.75MB) 
--------------------------------------------------
Total		 41.19M  	(157.13MB) 


In [ ]:
torch.autograd.set_detect_anomaly(True)
asr.train_single(df, 8, 1, False)

[Epoch 0/1]  (0 Steps in Total)
CTCNetwork:	 tensor(188.4495, device='cuda:0', grad_fn=<MeanBackward0>)
torch.Size([8, 436, 2001]) torch.Size([8, 77, 2001]) torch.Size([8, 436, 2001])
76 436


RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.cuda.FloatTensor [8]], which is output 0 of AsStridedBackward0, is at version 33649; expected version 33648 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).

In [ ]:
U = 5
T = 4
alpha = torch.zeros((U+1), T+1)

# The First U anti-diagonals that include the left most column's entries
for i in range(1, U):
    for j in range(min(i+1, T+1)):
        u, t = i-j, j
        alpha[u][t]
        print((u,t), end='\t')
    print()

# The remaining T anti-diagonals that include the top most row's entries (excluding the primary anti-diagoanal)        
for i in range(T+1):
    u, t = U, i
    while t <= T and u >= 0:
        alpha[u][t]
        print((u,t), end='\t')
        t += 1
        u -= 1
    print()


(1, 0)	(0, 1)	
(2, 0)	(1, 1)	(0, 2)	
(3, 0)	(2, 1)	(1, 2)	(0, 3)	
(4, 0)	(3, 1)	(2, 2)	(1, 3)	(0, 4)	
(5, 0)	(4, 1)	(3, 2)	(2, 3)	(1, 4)	
(5, 1)	(4, 2)	(3, 3)	(2, 4)	
(5, 2)	(4, 3)	(3, 4)	
(5, 3)	(4, 4)	
(5, 4)	
